In [13]:
import pandas as pd
import numpy as np
import json
from datasets import load_dataset

In [5]:
dataset = load_dataset('json', data_files='/kaggle/input/datasets/rayyankauchali0/resume-dataset/resumes_dataset.jsonl')

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
print(dataset['train'][0])

{'ResumeID': 'REAL_0001', 'Category': 'Java Developer', 'Name': 'Chad Griffin', 'Email': 'contact@email.com', 'Phone': '94105 555 4321000          10                     2014                                    102011 112013                        092008 102011                  2008 092008             2007 2008               052005 2007                     072003 052005                        2005', 'Location': 'City, State', 'Summary': 'jessica claire montgomery street san francisco ca 94105 555 4321000 resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design devel...', 'Skills': 'Python, SQL, Git, Linux', 'Experience': 'jessica claire montgomery street san francisco ca 94105 555 4321000 resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java

In [8]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['ResumeID', 'Category', 'Name', 'Email', 'Phone', 'Location', 'Summary', 'Skills', 'Experience', 'Education', 'Text', 'Source'],
        num_rows: 3500
    })
})


In [9]:
print(dataset['train'].features)

{'ResumeID': Value('string'), 'Category': Value('string'), 'Name': Value('string'), 'Email': Value('string'), 'Phone': Value('string'), 'Location': Value('string'), 'Summary': Value('string'), 'Skills': Value('string'), 'Experience': Value('string'), 'Education': Value('string'), 'Text': Value('string'), 'Source': Value('string')}


In [10]:
SYSTEM_PROMPT = """You are an expert HR parsing AI. Extract candidate information from the raw resume text and output it as a structured JSON object. Do not include any conversational filler."""

In [11]:
def format_for_finetuning(example):
    # 1. Clean the raw input text (remove extra spaces/newlines)
    raw_text = example.get('Text', '')
    if raw_text:
        raw_text = " ".join(raw_text.split())
    else:
        raw_text = ""

    # 2. Assemble the target JSON structure from the labeled features
    target_output = {
        "candidate_name": example.get('Name', ''),
        "contact_info": {
            "email": example.get('Email', ''),
            "phone": example.get('Phone', ''),
            "location": example.get('Location', '')
        },
        "professional_details": {
            "category": example.get('Category', ''),
            "skills": example.get('Skills', ''),
            "education": example.get('Education', ''),
            "experience": example.get('Experience', '')
        }
    }

    # 3. Create the conversational format required by most SFT trainers
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Extract the profile from this resume text:\n\n{raw_text}"},
        {"role": "assistant", "content": json.dumps(target_output)}
    ]

    # Return the new format. We will drop the old columns in the map function.
    return {"messages": messages}

In [14]:
processed_dataset = dataset.map(
    format_for_finetuning, 
    remove_columns=dataset['train'].column_names 
)

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

In [15]:
print(processed_dataset['train'][0])

{'messages': [{'content': 'You are an expert HR parsing AI. Extract candidate information from the raw resume text and output it as a structured JSON object. Do not include any conversational filler.', 'role': 'system'}, {'content': 'Extract the profile from this resume text:\n\njessica claire montgomery street san francisco ca 94105 555 4321000 resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible code team environment implemented designs including experimentation multiple iterations system administrator 102011 112013 mantech international corporation joint base mcguire nj technical lead sy

In [16]:
cleaned_dataset = processed_dataset.filter(
    lambda x: len(x['messages'][1]['content']) > 100 # Ensure user text isn't empty/tiny
    and '"candidate_name": ""' not in x['messages'][2]['content'] # Ensure we have a label
)

Filter:   0%|          | 0/3500 [00:00<?, ? examples/s]

In [17]:
print(f"Final training records: {len(cleaned_dataset['train'])}")

Final training records: 3500


In [18]:
print(cleaned_dataset['train'][0])

{'messages': [{'content': 'You are an expert HR parsing AI. Extract candidate information from the raw resume text and output it as a structured JSON object. Do not include any conversational filler.', 'role': 'system'}, {'content': 'Extract the profile from this resume text:\n\njessica claire montgomery street san francisco ca 94105 555 4321000 resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable extensible code team environment implemented designs including experimentation multiple iterations system administrator 102011 112013 mantech international corporation joint base mcguire nj technical lead sy

In [19]:
def is_valid_match(example):
    try:
        # 1. Extract the raw text and the target JSON string
        raw_text = example['messages'][1]['content'].lower()
        target_json_str = example['messages'][2]['content']
        
        # 2. Parse the target JSON back into a Python dictionary
        target_dict = json.loads(target_json_str)
        
        # 3. Extract the first name
        candidate_name = target_dict.get("candidate_name", "")
        if not candidate_name:
            return False # Drop rows with no name
            
        first_name = candidate_name.split()[0].lower()
        
        # 4. Check if the name is completely missing from the resume text
        if first_name not in raw_text:
            return False # Mismatch found! Drop the row.
            
        return True # It's a good row, keep it.
        
    except Exception as e:
        # If anything breaks (like bad JSON formatting), drop the row to be safe
        return False

In [20]:
cleaned_dataset = processed_dataset.filter(is_valid_match)

Filter:   0%|          | 0/3500 [00:00<?, ? examples/s]

In [21]:
original_size = len(processed_dataset['train'])
new_size = len(cleaned_dataset['train'])
dropped_count = original_size - new_size

print(f"Original Dataset Size: {original_size}")
print(f"Cleaned Dataset Size: {new_size}")
print(f"Rows Dropped (Poisoned Data): {dropped_count}")

Original Dataset Size: 3500
Cleaned Dataset Size: 1223
Rows Dropped (Poisoned Data): 2277


In [22]:
# Search the cleaned dataset to see if "Chad" survived
chad_survivors = [
    row for row in cleaned_dataset['train'] 
    if "Chad Griffin" in row['messages'][2]['content']
]

print(f"Number of 'Chad Griffin' rows remaining: {len(chad_survivors)}")

Number of 'Chad Griffin' rows remaining: 0


In [23]:
cleaned_dataset['train'].to_json("cleaned_resume_finetune_data.jsonl")

print("Dataset saved successfully!")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Dataset saved successfully!
